# 4장. pandas로 데이터에 질문하기

이 노트북은 데이터를 선택·필터링·정렬하고, 안전하게 병합한 뒤 완료 주문 기준 요약표를 만드는 실습 자료입니다.


## 0. 제출 정보

- 이름: 김용준
- GitHub ID: rotcev40
- 작성일: 2026-10-06
- 최종 제출 URL: https://github.com/rotcev40/llm-data-analysis-study/blob/main/chapter04/chapter04.ipynb

## 학습 목표

- 실제 컬럼명과 값의 종류를 확인한 뒤 필터링합니다.
- 수량과 단가로 주문 상세 금액을 계산합니다.
- `validate`와 `indicator`를 사용해 병합 결과를 검증합니다.
- 취소·환불 주문을 제외하고 완료 주문 기준 매출을 계산합니다.
- 개인정보를 최소화한 분석 결과를 저장합니다.


## 1. 프로젝트 루트와 실행 환경 확인

VS Code에서 Notebook을 실행하면 현재 작업 폴더가 프로젝트 루트 또는 `notebooks` 폴더일 수 있습니다. 아래 코드는 상위 폴더를 확인해 프로젝트 루트를 찾습니다.


In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Python 실행 파일:', sys.executable)
print('현재 작업 폴더:', Path.cwd())
print('프로젝트 루트:', PROJECT_ROOT)
print('데이터 폴더:', DATA_DIR)
print('결과 저장 폴더:', REPORT_DIR)


Python 실행 파일: C:\Users\yjun0\Desktop\서울대\4학년\2학기\LLM을 이용한 빅데이터 분석\llm-data-analysis-course\.venv\Scripts\python.exe
현재 작업 폴더: C:\Users\yjun0\Desktop\서울대\4학년\2학기\LLM을 이용한 빅데이터 분석\llm-data-analysis-course\notebooks
프로젝트 루트: C:\Users\yjun0\Desktop\서울대\4학년\2학기\LLM을 이용한 빅데이터 분석\llm-data-analysis-course
데이터 폴더: C:\Users\yjun0\Desktop\서울대\4학년\2학기\LLM을 이용한 빅데이터 분석\llm-data-analysis-course\data\raw
결과 저장 폴더: C:\Users\yjun0\Desktop\서울대\4학년\2학기\LLM을 이용한 빅데이터 분석\llm-data-analysis-course\reports


## 2. 데이터 파일 확인과 불러오기

파일이 없다면 프로젝트 루트에서 `python scripts/generate_sample_data.py`를 먼저 실행하세요.


In [2]:
required_files = ['customers.csv', 'products.csv', 'orders.csv', 'order_items.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        '필요한 데이터 파일이 없습니다: ' + ', '.join(missing_files)
        + '. 프로젝트 루트에서 python scripts/generate_sample_data.py를 실행하세요.'
    )

customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')

print('데이터 불러오기 완료')


데이터 불러오기 완료


## 3. 데이터 구조와 필수 컬럼 확인

LLM이 작성한 코드가 실제 컬럼명과 일치하는지 먼저 확인합니다.


In [3]:
datasets = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

expected_columns = {
    'customers': ['customer_id', 'gender', 'age', 'city'],
    'products': ['product_id', 'product_name', 'category', 'price'],
    'orders': ['order_id', 'customer_id', 'order_date', 'order_status'],
    'order_items': ['order_id', 'product_id', 'quantity', 'unit_price'],
}

for name, df in datasets.items():
    missing = [col for col in expected_columns[name] if col not in df.columns]
    print(name, df.shape, df.columns.tolist())
    if missing:
        raise KeyError(f'{name}에 필요한 컬럼이 없습니다: {missing}')


customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (300, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


## 4. 컬럼 선택과 행 필터링

이 저장소의 샘플 데이터는 도시명을 `서울`, `부산`처럼 한글로 생성합니다. 필터링 전에 실제 값을 확인합니다.


In [4]:
customer_basic = customers[['customer_id', 'gender', 'age', 'city']]
display(customer_basic.head())

print('도시별 고객 수')
display(customers['city'].value_counts())

customers_over_30 = customers[customers['age'] >= 30]
seoul_customers = customers[customers['city'] == '서울']
city_customers = customers[customers['city'].isin(['서울', '부산'])]

print('30세 이상 고객 수:', len(customers_over_30))
print('서울 고객 수:', len(seoul_customers))
print('서울 또는 부산 고객 수:', len(city_customers))


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


도시별 고객 수


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

30세 이상 고객 수: 111
서울 고객 수: 15
서울 또는 부산 고객 수: 31


## 5. 정렬과 주문 상태 확인


In [5]:
display(products.sort_values('price', ascending=False).head(10))
display(customers.sort_values('age', ascending=False).head(10))

print('주문 상태별 건수')
display(orders['order_status'].value_counts())


,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


,customer_id,name,gender,age,city,signup_date
14,15,장정식,M,69,서울,2026-08-12
8,9,송지민,M,69,서울,2025-12-27
54,55,윤영철,M,69,광주,2023-10-08
78,79,이서준,M,69,부산,2023-09-21
136,137,김선영,M,68,울산,2025-02-05
45,46,김서현,M,67,대전,2026-07-21
132,133,김성진,M,67,성남,2026-08-06
123,124,김시우,M,67,대구,2024-12-02
11,12,김정남,F,66,대전,2024-12-08
57,58,김성훈,F,66,성남,2025-08-22


주문 상태별 건수


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 과제 1. 질문과 필요한 데이터 선택

### 내가 확인하려는 질문

어떤 상품 카테고리의 completed 주문 기준 금액이 가장 큰가?

### 사용한 파일/컬럼

- order_items: order_id, product_id, quantity, unit_price
- orders: order_id, customer_id, order_date, order_status
- products: product_id, product_name, category
- 금액 계산식: line_total = quantity * unit_price
- 포함할 주문 상태: completed
- 최종 집계 기준: category별 total_sales 합계

### 결과 관찰

order_status의 실제 값은 completed/cancelled/refunded 3종이고 completed는 184건이다. category 컬럼은 products에만 있고 order_items에는 없다.

### 나의 해석과 판단

금액을 묻는 질문이므로 취소/환불 주문은 빼야 한다고 생각했다. category가 order_items에 없기 때문에 products를 병합하지 않으면 카테고리별 집계 자체가 되지 않는다.

### 업무·분석적 의미

집계 전에 범위를 정해두면 나중에 숫자가 달라졌을 때 어디서 갈라졌는지 찾기 쉬울 것 같다.

### 한계와 추가 확인 사항

completed가 실제로 결제와 배송까지 끝난 상태를 뜻하는지는 데이터만으로 알 수 없다.

## 6. 주문 상세 금액 만들기

`line_total`은 주문 상세 1행의 금액입니다. 주문 상태를 연결하기 전 합계는 확정 매출이 아니라 전체 주문 상세 금액입니다.


In [6]:
order_items = order_items.copy()
order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

all_order_amount = order_items['line_total'].sum()
print('전체 주문 상세 금액:', all_order_amount)
display(order_items[['order_id', 'product_id', 'quantity', 'unit_price', 'line_total']].head())


전체 주문 상세 금액: 255610000


,order_id,product_id,quantity,unit_price,line_total
0,1,100,3,102000,306000
1,1,87,5,25000,125000
2,1,7,3,142000,426000
3,1,9,3,193000,579000
4,2,72,4,189000,756000


## 7. 주문 데이터 병합과 검증

`validate='many_to_one'`은 주문 상세의 동일 주문 ID는 여러 번 나올 수 있지만 주문 테이블의 주문 ID는 한 번만 나와야 한다는 뜻입니다.


In [7]:
print('orders.order_id 중복 수:', orders['order_id'].duplicated().sum())

order_sales = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(order_items))
print('병합 후 행 수:', len(order_sales))
display(order_sales['_merge'].value_counts())

order_sales = order_sales.drop(columns='_merge')


orders.order_id 중복 수: 0
병합 전 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

## 8. 완료 주문만 선택하기

이후의 매출 요약은 `order_status == 'completed'`인 주문만 사용합니다.


In [8]:
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
print('날짜 변환 실패:', order_sales['order_date'].isna().sum())

completed_order_sales = order_sales[
    order_sales['order_status'] == 'completed'
].copy()

completed_order_sales['order_month'] = (
    completed_order_sales['order_date'].dt.to_period('M').astype(str)
)

print('전체 주문 상세 행 수:', len(order_sales))
print('완료 주문 상세 행 수:', len(completed_order_sales))
print('완료 주문 매출:', completed_order_sales['line_total'].sum())


날짜 변환 실패: 0
전체 주문 상세 행 수: 764
완료 주문 상세 행 수: 474
완료 주문 매출: 148990000


## 과제 2. 필터링·정렬·파생 컬럼

- 적용한 필터 조건: `order_status == 'completed'`
- 정렬 기준: total_sales 내림차순
- 만든 파생 컬럼: line_total, order_month
- `line_total` 계산식: quantity * unit_price

### Evidence

![필터와 파생 컬럼](images/step02_transform.png)

### 결과 관찰

line_total을 만든 뒤 전체 764행의 합계는 255610000이었다. completed만 남기면 474행에 148990000이다. 날짜 변환 실패는 0건이다.

### 나의 해석과 판단

필터를 걸지 않으면 금액이 1.72배가 된다. 취소/환불 주문에도 order_items가 그대로 붙어 있기 때문이다.

### 업무·분석적 의미

같은 데이터라도 범위를 어떻게 잡느냐에 따라 답이 달라지므로, 숫자를 보고하기 전에 범위를 먼저 밝혀야 할 것 같다.

### 한계와 추가 확인 사항

refunded를 금액에서 빼는 것이 맞는지는 회계 기준에 따라 다를 수 있다.

## 9. 상품 데이터 병합과 검증


In [9]:
print('products.product_id 중복 수:', products['product_id'].duplicated().sum())

completed_sales_items = completed_order_sales.merge(
    products,
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(completed_order_sales))
print('병합 후 행 수:', len(completed_sales_items))
display(completed_sales_items['_merge'].value_counts())

completed_sales_items = completed_sales_items.drop(columns='_merge')


products.product_id 중복 수: 0
병합 전 행 수: 474
병합 후 행 수: 474


_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64

## 과제 3. merge 검증

- 병합한 데이터: order_items + orders, 이어서 completed_order_sales + products
- 사용한 key: order_id, product_id
- `validate` 결과: 두 병합 모두 many_to_one 통과(오류 없음)
- `indicator` 결과: 첫 병합 both 764건, 두 번째 both 474건, left_only와 right_only는 모두 0건
- 병합 전/후 행 수: 764 -> 764, 474 -> 474

### Evidence

![merge 검증](images/step03_merge.png)

### 결과 관찰

두 병합 모두 행 수가 변하지 않았고 미매칭이 0건이었다. orders.order_id와 products.product_id의 중복도 각각 0건이다.

### 나의 해석과 판단

오른쪽 테이블의 key가 유일하고 validate='many_to_one'이 통과했으므로 행이 불어날 수 없다고 판단했다. 미매칭이 0건이라 빠진 주문이나 상품도 없다.

### 업무·분석적 의미

key가 중복이면 merge에서 행이 늘어나 금액이 부풀려지는데, 집계 결과만 보면 알아채기 어렵다.

### 한계와 추가 확인 사항

행 수가 유지된다는 것과 붙은 값이 맞다는 것은 다른 문제다.

## 10. 카테고리별·상품별 매출


In [10]:
category_sales = (
    completed_sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

display(category_sales)

product_sales = (
    completed_sales_items
    .groupby(['product_id', 'product_name', 'category'], as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

display(product_sales.head(10))


,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


,product_id,product_name,category,total_quantity,total_sales
39,41,스포츠 상품 041,스포츠,35,5705000
11,12,식품 상품 012,식품,25,4375000
8,9,스포츠 상품 009,스포츠,20,3860000
70,72,뷰티 상품 072,뷰티,20,3780000
69,71,전자기기 상품 071,전자기기,23,3703000
66,68,스포츠 상품 068,스포츠,26,3640000
78,81,전자기기 상품 081,전자기기,22,3630000
10,11,패션 상품 011,패션,31,3565000
20,22,생활용품 상품 022,생활용품,29,3248000
86,89,생활용품 상품 089,생활용품,30,3090000


## 11. 월별 매출


In [11]:
monthly_summary = (
    completed_order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_summary['average_order_value'] = (
    monthly_summary['total_sales'] / monthly_summary['order_count']
).round(0)

display(monthly_summary)


,order_month,total_sales,order_count,average_order_value
0,2025-09,2650000,5,530000.0
1,2025-10,16970000,19,893158.0
2,2025-11,11021000,13,847769.0
3,2025-12,21085000,23,916739.0
4,2026-01,14272000,16,892000.0
5,2026-02,4784000,7,683429.0
6,2026-03,18323000,25,732920.0
7,2026-04,10254000,15,683600.0
8,2026-05,16831000,19,885842.0
9,2026-06,12994000,17,764353.0


## 12. 고객별 구매 금액

고객 ID로 먼저 집계한 뒤 고객 속성을 붙입니다. 출력에는 실제 이름 대신 익명화된 고객 라벨을 사용합니다.


In [12]:
customer_sales = (
    completed_order_sales
    .groupby('customer_id', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales = customer_sales.merge(
    customers[['customer_id', 'city']],
    on='customer_id',
    how='left',
    validate='one_to_one',
)

customer_sales['customer_label'] = 'Customer ' + customer_sales['customer_id'].astype(str)
display(customer_sales[['customer_label', 'city', 'order_count', 'total_sales']].head(10))


,customer_label,city,order_count,total_sales
0,Customer 117,성남,5,4100000
1,Customer 102,고양,4,3996000
2,Customer 83,수원,4,3880000
3,Customer 30,서울,5,3590000
4,Customer 40,서울,4,3523000
5,Customer 20,인천,2,3191000
6,Customer 3,성남,2,3178000
7,Customer 111,광주,3,3153000
8,Customer 66,서울,4,3093000
9,Customer 147,부산,2,2990000


## 과제 4. completed 주문 범위와 집계

- 분석 범위 정의: `order_status == 'completed'`인 주문 상세 474행
- 카테고리별 결과: 스포츠 31743000(21.31%) > 전자기기 26400000 > 생활용품 23915000 > 뷰티 23383000 > 식품 16573000 > 도서 16389000 > 패션 10587000
- 상품별 결과: 스포츠 상품 041이 5705000으로 1위
- 월별 결과: 2025-12가 21085000으로 가장 크고 2026-09가 4051000으로 가장 작다
- 고객별 결과: Customer 117(성남)이 5건 4100000으로 1위

### Evidence

![핵심 집계 결과](images/step04_groupby.png)

### 결과 관찰

질문의 답은 스포츠다. 수량도 295개로 가장 많다. 생활용품은 수량이 272개로 2위인데 금액은 3위이고, 식품은 수량이 133개로 적은 편인데 금액은 5위다. 월별로는 2026-08의 주문이 5건뿐인데 평균 주문 금액은 1020600으로 가장 높다.

### 나의 해석과 판단

카테고리별 결과가 가장 중요하다고 생각한다. 질문이 그것을 묻기 때문이다. 수량 순위와 금액 순위가 어긋나는 카테고리가 있는 것으로 보아 단가 차이가 영향을 주는 것 같다.

### 업무·분석적 의미

수량과 금액을 같이 봐야 싸게 많이 팔리는 카테고리와 비싸게 적게 팔리는 카테고리를 구분할 수 있다.

### 한계와 추가 확인 사항

total_sales는 completed 주문의 quantity * unit_price 합계일 뿐이다. 할인, 배송비, 반품, 원가가 반영되지 않았으므로 회계상 순매출과 같다고 볼 수 없다. 월별 수치도 양 끝 달이 보름치만 들어 있어 그대로 비교하기 어렵다.

## 12-1. 총합 일치 검증

집계 기준을 바꿔도 completed 주문의 금액 합계는 같아야 합니다.
원본 합계와 카테고리별·상품별·월별·고객별 합계를 한 표에서 비교합니다.

In [13]:
base_total = completed_order_sales['line_total'].sum()

total_check = pd.DataFrame([
    {'기준': '원본 completed line_total', 'total_sales': base_total},
    {'기준': '카테고리별 합계', 'total_sales': category_sales['total_sales'].sum()},
    {'기준': '상품별 합계', 'total_sales': product_sales['total_sales'].sum()},
    {'기준': '월별 합계', 'total_sales': monthly_summary['total_sales'].sum()},
    {'기준': '고객별 합계', 'total_sales': customer_sales['total_sales'].sum()},
])

total_check['원본과의 차이'] = total_check['total_sales'] - base_total
total_check['일치'] = total_check['원본과의 차이'] == 0

display(total_check)
print('모든 집계 기준 합계 일치:', total_check['일치'].all())

,기준,total_sales,원본과의 차이,일치
0,원본 completed line_total,148990000,0,True
1,카테고리별 합계,148990000,0,True
2,상품별 합계,148990000,0,True
3,월별 합계,148990000,0,True
4,고객별 합계,148990000,0,True


모든 집계 기준 합계 일치: True


## 과제 5. 총합 일치 검증

- 원본 completed `line_total` 합계: 148990000
- 카테고리 합계: 148990000
- 월별 합계: 148990000
- 고객별 합계: 148990000
- 차이 여부: 상품별 합계까지 포함해 5개 기준 모두 차이 0

### Evidence

![총합 검증](images/step05_total_check.png)

### 나의 해석과 판단

집계 기준을 바꿔도 같은 범위를 썼다면 합계는 같아야 한다. 합계가 다르다면 중간에 필터가 달라졌거나 merge에서 행이 늘거나 줄었다는 뜻이므로, 필터 범위, merge 행 수, groupby 키 순서로 확인하면 될 것 같다.

## 13. 결과 저장하기


In [14]:
outputs = {
    'ch04_category_sales.csv': category_sales,
    'ch04_product_sales.csv': product_sales,
    'ch04_monthly_sales.csv': monthly_summary,
    'ch04_customer_sales.csv': customer_sales,
}

for filename, df in outputs.items():
    path = REPORT_DIR / filename
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(filename, path.exists(), path.stat().st_size)


ch04_category_sales.csv True 254
ch04_product_sales.csv True 4409
ch04_monthly_sales.csv True 443
ch04_customer_sales.csv True 3387


## 14. LLM 코드 검증 연습

```text
LLM이 다음 코드를 제안했습니다.

category_sales = order_items.groupby('category')['line_total'].sum()

현재 데이터에서 이 코드가 바로 실행 가능한지 검토해 주세요.
category 컬럼의 위치, 주문 상태 필터링, merge validate와 indicator를 포함해
안전한 수정 코드와 검증 순서를 설명해 주세요.
```


## 과제 6. LLM pandas 코드 검증

- LLM Prompt 요약: `order_items.groupby('category')['line_total'].sum()`이 현재 데이터에서 바로 실행되는지 검토해 달라고 요청했다. 4개 파일의 컬럼 목록과 order_status 값 3종을 함께 제공했다.
- 제안 코드 요약: 기본키 중복과 결측을 먼저 확인하고, line_total을 만든 뒤 orders와 products를 `validate='many_to_one'`, `indicator`로 두 번 병합하고, completed만 남겨 카테고리별로 합산하는 순서를 제시했다.
- 실제 컬럼/범위와 맞지 않은 부분: 원래 코드의 category와 line_total이 order_items에 없다는 지적은 맞았다. 다만 unit_price를 주문 당시 적용된 단가라서 products.price와 다를 수 있다고 설명했는데, 이 데이터에서는 764행 전부 두 값이 같다.
- 수정한 내용: merge의 validate와 indicator, completed 필터, 합계 검증은 그대로 적용했다. 제안에 있던 order_item_id 중복 확인과 quantity/unit_price 음수 확인은 3장에서 이미 본 항목이라 이번 노트북에는 넣지 않았다.
- 최종 판단: 수정 후 사용

### Evidence

![LLM 코드 검증](images/step06_llm_validation.png)

### 나의 해석과 판단

원래 코드는 실행하면 KeyError가 나기 때문에 틀렸다는 것을 바로 알 수 있다. 문제는 completed 필터를 빠뜨린 코드다. 그런 코드는 오류 없이 돌아가고 255610000이라는 숫자를 내놓는다. 틀린 범위로 계산해도 결과가 나오기 때문에 숫자가 나왔다는 것만으로는 맞는지 알 수 없다. unit_price에 대한 설명처럼 LLM이 컬럼 이름만 보고 역할을 추측하는 경우도 있어서 실제 값으로 확인해야 할 것 같다.

## 15. 실습 과제

1. 40세 이상 고객을 추출합니다.
2. 상품 가격이 낮은 순서대로 10개를 출력합니다.
3. 결제수단별 주문 수와 비율을 계산합니다.
4. 카테고리별 평균 상품 가격을 계산합니다.
5. 완료 주문과 전체 주문의 금액 차이를 계산합니다.
6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트를 작성합니다.


In [15]:
# 과제 코드를 아래에 작성하세요.

# 1. 40세 이상 고객 추출
customers_40plus = customers[customers['age'] >= 40]
print('40세 이상 고객 수:', len(customers_40plus), '/', len(customers))
display(customers_40plus[['customer_id', 'gender', 'age', 'city']].head())

40세 이상 고객 수: 78 / 150


,customer_id,gender,age,city
2,3,F,61,성남
3,4,F,55,울산
6,7,F,53,인천
8,9,M,69,서울
9,10,F,62,울산


In [16]:
# 2. 상품 가격이 낮은 순서대로 10개
display(products.sort_values('price')[['product_id', 'product_name', 'category', 'price']].head(10))

,product_id,product_name,category,price
5,6,전자기기 상품 006,전자기기,5000
27,28,뷰티 상품 028,뷰티,5000
97,98,스포츠 상품 098,스포츠,10000
46,47,생활용품 상품 047,생활용품,11000
94,95,전자기기 상품 095,전자기기,20000
49,50,뷰티 상품 050,뷰티,23000
82,83,전자기기 상품 083,전자기기,24000
86,87,도서 상품 087,도서,25000
41,42,패션 상품 042,패션,28000
58,59,뷰티 상품 059,뷰티,32000


In [17]:
# 3. 결제수단별 주문 수와 비율
payment_summary = (
    orders.groupby('payment_method', as_index=False)
    .agg(order_count=('order_id', 'nunique'))
    .sort_values('order_count', ascending=False)
)
payment_summary['ratio(%)'] = (
    payment_summary['order_count'] / payment_summary['order_count'].sum() * 100
).round(2)

display(payment_summary)

,payment_method,order_count,ratio(%)
2,kakao_pay,79,26.33
3,naver_pay,77,25.67
0,bank_transfer,74,24.67
1,card,70,23.33


In [18]:
# 4. 카테고리별 평균 상품 가격
category_price = (
    products.groupby('category', as_index=False)
    .agg(product_count=('product_id', 'count'), avg_price=('price', 'mean'))
    .sort_values('avg_price', ascending=False)
)
category_price['avg_price'] = category_price['avg_price'].round(0)

display(category_price)

,category,product_count,avg_price
4,식품,7,137143.0
1,뷰티,16,117688.0
6,패션,11,115909.0
3,스포츠,19,111579.0
0,도서,14,106857.0
5,전자기기,17,101588.0
2,생활용품,16,96438.0


In [19]:
# 5. 완료 주문과 전체 주문의 금액 차이
all_total = order_sales['line_total'].sum()
completed_total = completed_order_sales['line_total'].sum()

print('전체 주문 금액:', all_total)
print('완료 주문 금액:', completed_total)
print('차이:', all_total - completed_total)
print('완료 주문 비중(%):', round(completed_total / all_total * 100, 2))

display(
    order_sales.groupby('order_status', as_index=False)
    .agg(order_count=('order_id', 'nunique'), total_sales=('line_total', 'sum'))
    .sort_values('total_sales', ascending=False)
)

전체 주문 금액: 255610000
완료 주문 금액: 148990000
차이: 106620000
완료 주문 비중(%): 58.29


,order_status,order_count,total_sales
1,completed,184,148990000
0,cancelled,64,62181000
2,refunded,52,44439000


In [20]:
# 6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트
task6_prompt = """
pandas로 고객별 구매 금액을 집계하는 코드를 작성해 주세요.

데이터는 4개 CSV입니다.
- customers: customer_id, name, gender, age, city, signup_date
- products: product_id, product_name, category, price
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_item_id, order_id, product_id, quantity, unit_price

조건
- 금액은 quantity * unit_price로 계산합니다.
- order_status가 completed인 주문만 포함합니다.
- 주문 건수는 order_id.nunique()로 셉니다.
- merge에는 validate와 indicator를 넣어 검증 결과를 출력해 주세요.
- 집계 후 전체 합계가 원본 합계와 같은지 확인하는 코드도 함께 작성해 주세요.

실제로 존재하는 컬럼만 사용하고, 데이터로 확인하지 않은 내용은 단정하지 말아 주세요.
"""

print(task6_prompt)


pandas로 고객별 구매 금액을 집계하는 코드를 작성해 주세요.

데이터는 4개 CSV입니다.
- customers: customer_id, name, gender, age, city, signup_date
- products: product_id, product_name, category, price
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_item_id, order_id, product_id, quantity, unit_price

조건
- 금액은 quantity * unit_price로 계산합니다.
- order_status가 completed인 주문만 포함합니다.
- 주문 건수는 order_id.nunique()로 셉니다.
- merge에는 validate와 indicator를 넣어 검증 결과를 출력해 주세요.
- 집계 후 전체 합계가 원본 합계와 같은지 확인하는 코드도 함께 작성해 주세요.

실제로 존재하는 컬럼만 사용하고, 데이터로 확인하지 않은 내용은 단정하지 말아 주세요.



## 정리

이번 장에서는 실제 값 확인, 컬럼 선택, 필터링, 정렬, 파생 컬럼, 병합 검증, 완료 주문 기준 집계, CSV 저장 과정을 수행했습니다. 다음 장에서는 결측치, 중복, 타입 오류, 날짜 형식, 이상값 후보를 다루는 데이터 전처리로 이어집니다.


## 과제 7. Chapter 04 최종 인사이트

### 가장 의미 있다고 생각한 결과 2가지

1. 스포츠 카테고리가 31743000으로 completed 기준 금액 1위이며 전체의 21.31%를 차지한다.
2. completed만 보면 148990000인데 전체 주문을 다 넣으면 255610000이 되어 1.72배 차이가 난다.

### 그 결과를 뒷받침하는 수치/표

카테고리별 집계표와 총합 일치 검증표다. 5개 집계 기준이 모두 148990000으로 같아서 범위가 중간에 바뀌지 않았다는 것을 확인했다.

### 추가로 확인하고 싶은 질문

카테고리별로 취소/환불 비율이 다른지 궁금하다. 스포츠가 completed에서 1위인데 취소까지 포함하면 순위가 달라지는지 확인해보고 싶다.

### 현재 결과의 한계

1년치 데이터이고 양 끝 달은 보름치만 들어 있다. total_sales도 매출이라고 부르기에는 빠진 항목이 많다.

## 최종 제출 체크

- [x] 핵심 셀 Output이 남아 있습니다.
- [x] merge와 총합 검증 Evidence가 있습니다.
- [x] 결과 관찰과 해석이 구분되어 있습니다.
- [x] LLM 코드를 검증했습니다.
- [x] 개인정보/Secret이 없습니다.
- [x] `chapter04/chapter04.ipynb`가 GitHub에서 정상 표시됩니다.
- [x] 최종 Notebook 파일 URL을 제출합니다.